# 🔍 Reflexion Agents: Self-Critique Plus External Evidence

## Learning Objectives
In this notebook, you will learn:
1. **The Reflexion pattern** - How it extends reflection with a structured self-critique and research queries
2. **A structured responder** - Force the answer, critique and queries into one tool-call schema, with a retry that feeds validation errors back
3. **Research and revision** - Run the model's own search queries, then revise the answer with numbered citations
4. **The loop as a graph** - Draft → search → revise, bounded by a revision count, with the result checked in code
5. **A second model family (optional)** - Run the same responder on Qwen via Databricks

## Where this fits
`01_reflection_agents.ipynb` let a critic review drafts with nothing but its own opinion - and when the critic asked for "a statistic", the writer made one up. Reflexion makes the critique **name what is missing** and turns that into **web searches**, so the revision can cite evidence instead of inventing it. `03_reflection_architecture.ipynb` returns to a single-pass reflection, but with a structured critique and an objective test.

## Prerequisites
- `01_reflection_agents.ipynb` (reflection with messages in LangGraph)
- `ToolNode` - `modules/02_tools_and_function_calling/notebooks/03_augmented_llm_with_tools_langgraph.ipynb`
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` and `TAVILY_API_KEY` in the project-root `.env`
- Optional: `DATABRICKS_HOST` and `DATABRICKS_TOKEN` for Part 6 (Qwen via Databricks Foundation Model APIs) - skipped cleanly when absent

---
## 📖 Part 1: The Reflexion Pattern

The [Reflexion pattern](https://arxiv.org/pdf/2303.11366), introduced by Shinn et al., extends basic reflection by combining **self-critique** with **external knowledge integration** and **structured output parsing**. Unlike simple reflection, Reflexion allows an agent to learn from mistakes in real time while leveraging additional information.

The workflow typically follows these steps:
- **Initial Generation:** The agent produces a response along with self-critique and research queries.
- **External Research:** Knowledge gaps identified during critique trigger web searches or other information retrieval.
- **Knowledge Integration:** New insights are incorporated into an improved response.
- **Iterative Refinement:** The agent repeats the cycle until the response meets desired quality thresholds.

| Property | Value |
|---|---|
| Origin | Shinn et al., *Reflexion: Language Agents with Verbal Reinforcement Learning* (2023). [arXiv:2303.11366](https://arxiv.org/abs/2303.11366) |

### Actor Architecture

At its core, a Reflexion agent is built around an **Actor** — an agent that generates an initial response, critiques it, and then re-executes the task with improvements. Supporting this loop are a few critical sub-components:

- **Tool execution:** Access to external knowledge sources.
- **Initial responder:** Generates the first draft along with self-reflection.
- **Revisor/Revision:** Produces refined outputs by incorporating prior reflections.

---
## 📦 Step 1: Environment Setup

Load the API keys, create the chat model through the course factory, and set up the Tavily search tool.

In [1]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY, TAVILY_API_KEY and (optionally) DATABRICKS_HOST / DATABRICKS_TOKEN from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

✅ Environment ready


In [2]:
# ============================================================================
# LLM INITIALIZATION: The Reflexion actor (responder and revisor)
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini")

print(f"✅ LLM initialized: {llm.model_name}")

✅ LLM initialized: gpt-4o-mini


In [3]:
# ============================================================================
# SEARCH TOOL: Tavily, used to research the gaps the critique names
# ============================================================================
from langchain_tavily import TavilySearch

tavily_tool = TavilySearch(max_results=3)
print(f"✅ Search tool ready: {tavily_tool.name} (max_results={tavily_tool.max_results})")

✅ Search tool ready: tavily_search (max_results=3)


---
## 🧱 Part 2: A Structured Responder

### 2.1 The Actor Prompt

The actor's prompt tells it to (1) answer, (2) reflect on and critique its own answer - "be severe" - and (3) recommend search queries that would improve it. The same template serves the first draft and every revision; only `primary_instruction` and the name of the function to call change.

In [4]:
# ============================================================================
# ACTOR PROMPT TEMPLATE: Shared by the responder and the revisor
# ============================================================================
import datetime

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

actor_prompt_template = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """You are an expert technical educator specializing in machine learning and neural networks.
                Current time: {time}
                1. {primary_instruction}
                2. Reflect and critique your answer. Be severe to maximize improvement.
                3. Recommend search queries to research information and improve your answer.""",
        ),
        MessagesPlaceholder(variable_name="messages"),
        (
            "user",
            "\n\n<reminder>Reflect on the user's original question and the"
            " actions taken thus far. Respond using the {function_name} function.</reminder>",
        ),
    ]
).partial(
    time=lambda: datetime.datetime.now().isoformat(),
)

print("✅ Actor prompt template ready")

✅ Actor prompt template ready


### 2.2 Enforce Structured Output

Two Pydantic models define the output:

- **`Reflection`** captures the self-critique: what information is **missing** and what is **superfluous**.
- **`GenerateResponse`** holds the answer, the reflection, and 1-3 `research_queries`.

We **bind `GenerateResponse` as a tool** (rather than using `with_structured_output`) on purpose: the answer then arrives as a tool call inside an `AIMessage`, which a `ToolNode` can later "execute" - by running the research queries.

In [5]:
# ============================================================================
# SCHEMAS: Answer + self-critique + research queries
# ============================================================================
from pydantic import BaseModel, Field, ValidationError

class Reflection(BaseModel):
    missing: str = Field(description="Critique of what is missing.")
    superfluous: str = Field(description="Critique of what is superfluous")

class GenerateResponse(BaseModel):
    """Generate response. Provide an answer, critique, and then follow up with search queries to improve the answer."""
    
    response: str = Field(description="~250 word detailed answer to the question.")
    reflection: Reflection = Field(description="Your reflection on the initial answer.")
    research_queries: list[str] = Field(
        description="1-3 search queries for researching improvements to address the critique of your current answer."
    )

print(f"✅ GenerateResponse fields: {list(GenerateResponse.model_fields)}")

✅ GenerateResponse fields: ['response', 'reflection', 'research_queries']


### 2.3 Retry with Schema Feedback

Structured parsing can fail if the output doesn't match the schema. `AdaptiveResponder` validates each response; on a `ValidationError` it appends the failed attempt plus a `ToolMessage` with the error and the schema, and asks again (up to 3 attempts). The original version appended to the *state dict* instead of the message list, so a retry would have crashed; it now extends the list of messages.

In [6]:
# ============================================================================
# ADAPTIVE RESPONDER: Validate the tool call, retry with the error as feedback
# ============================================================================
import json

from langchain_core.messages import HumanMessage, ToolMessage
from langchain_core.output_parsers.openai_tools import PydanticToolsParser


class AdaptiveResponder:
    def __init__(self, chain, output_parser, schema):
        self.chain = chain
        self.output_parser = output_parser
        self.schema = schema

    def generate(self, state: dict):
        messages = list(state["messages"])
        llm_response = None
        for retry_count in range(3):
            llm_response = self.chain.invoke({"messages": messages}, {"tags": [f"attempt:{retry_count}"]})
            try:
                self.output_parser.invoke(llm_response)
                return {"messages": llm_response}
            except ValidationError as validation_error:
                schema_json = json.dumps(self.schema.model_json_schema(), indent=2)
                messages = messages + [
                    llm_response,
                    ToolMessage(
                        content=f"{repr(validation_error)}\n\nPay close attention to the function schema.\n\n"
                        f"{schema_json}\n\nRespond by fixing all validation errors.",
                        tool_call_id=llm_response.tool_calls[0]["id"],
                    ),
                ]
        return {"messages": llm_response}


print("✅ AdaptiveResponder defined")

✅ AdaptiveResponder defined


### 2.4 🧪 The Initial Responder

Bind `GenerateResponse` as a tool, **force** the model to call it (`tool_choice`), and turn on OpenAI's **strict** schema mode, then wrap the chain in `AdaptiveResponder`. Strict mode matters: without it, `gpt-4o-mini` left out the required `sources` field of the revision schema in 3 of 4 test calls, often on every retry. The retry loop stays as the fallback for providers without a strict mode (such as the Qwen model on Databricks in Part 6). The test prints the three parts of the structured response.

In [7]:
# ============================================================================
# INITIAL RESPONDER: First draft + self-critique + research queries
# ============================================================================
def make_responder(model, schema, instruction, **bind_kwargs):
    chain = actor_prompt_template.partial(
        primary_instruction=instruction, function_name=schema.__name__
    ) | model.bind_tools(tools=[schema], tool_choice=schema.__name__, **bind_kwargs)
    return AdaptiveResponder(chain=chain, output_parser=PydanticToolsParser(tools=[schema]), schema=schema)


FIRST_DRAFT_INSTRUCTION = "Provide a detailed ~250 word explanation suitable for someone with basic programming background."
# strict=True: OpenAI enforces the JSON schema, so required fields cannot be left out
initial_responder = make_responder(llm, GenerateResponse, FIRST_DRAFT_INSTRUCTION, strict=True)

example_question = "What is the difference between supervised and unsupervised learning?"
initial = initial_responder.generate({"messages": [HumanMessage(content=example_question)]})

draft = initial["messages"].tool_calls[0]["args"]
print(f"📝 Answer ({len(draft['response'].split())} words): {draft['response'][:300]}...")
print(f"🔎 Missing: {draft['reflection']['missing']}")
print(f"✂️  Superfluous: {draft['reflection']['superfluous']}")
print(f"🌐 Research queries: {draft['research_queries']}")

📝 Answer (203 words): Supervised and unsupervised learning are two fundamental approaches in machine learning, each serving different purposes and utilizing different types of data.

**Supervised Learning** involves training a model on a labeled dataset, which means that each training example is paired with an output lab...
🔎 Missing: The explanation could benefit from more concrete examples and a clearer distinction between the types of problems each method addresses. Additionally, mentioning semi-supervised and reinforcement learning could provide a broader context.
✂️  Superfluous: Some technical jargon, like specific algorithms, might overwhelm a beginner. Simplifying the language and focusing on core concepts would enhance understanding.
🌐 Research queries: ['examples of supervised vs unsupervised learning', 'applications of supervised learning', 'applications of unsupervised learning', 'differences between supervised and unsupervised learning in machine learning']


---
## 🔬 Part 3: Research and Revision

The revision step combines three things - the original draft, the self-critique, and the research results - to produce an evidence-backed answer. The `improvement_guidelines` require numbered citations and a **References** section, and keep the ~250-word budget. Two lines were added after testing: with only the original guidelines, `gpt-4o-mini` returned the previous answer almost word for word with a source list attached but no citations in the text, and re-issued the same research queries - so every later revision was identical. The revisor also runs at temperature 0.7 - at temperature 0, a second revision given the same material simply repeated the first. `ImproveResponse` extends `GenerateResponse` with a `sources` field.

In [8]:
# ============================================================================
# REVISOR: Guidelines + ImproveResponse schema
# ============================================================================
improvement_guidelines = """Revise your previous explanation using the new information.
    - You should use the previous critique to add important technical details to your explanation.
    - You MUST include numerical citations in your revised answer to ensure it can be verified.
    - Add a "References" section to the bottom of your answer (which does not count towards the word limit).
    - For the references field, provide a clean list of URLs only (e.g., ["https://example.com", "https://example2.com"])
    - You should use the previous critique to remove superfluous information from your answer and make SURE it is not more than 250 words.
    - Keep the explanation accessible for someone with basic programming background while being technically accurate.
    - Do not copy your previous answer: add at least one concrete fact taken from the search results,
      and cite it in the text as [n], where n is the position of its URL in `sources` (starting at 1).
    - Your new research_queries must be different from every query already searched.
"""


class ImproveResponse(GenerateResponse):
    """Improve your original answer to your question. Provide an answer, reflection,
    cite your reflection with references, and finally
    add search queries to improve the answer."""
    
    sources: list[str] = Field(
        description="List of reference URLs that support your answer. Each reference should be a clean URL string."
    )


# temperature 0.7 for the revisor: at temperature 0 the second revision came back
# word-for-word identical to the first (same input -> same output), so the loop added nothing.
revisor_llm = get_openai_llm(model="gpt-4o-mini", temperature=0.7)
response_improver = make_responder(revisor_llm, ImproveResponse, improvement_guidelines, strict=True)
print(f"✅ Revisor ready (extra field: {set(ImproveResponse.model_fields) - set(GenerateResponse.model_fields)})")

✅ Revisor ready (extra field: {'sources'})


Test the revisor by hand: the question, the draft (as the `AIMessage` with its tool call), and a `ToolMessage` holding the search results for the draft's first research query.

In [9]:
# ============================================================================
# ONE REVISION BY HAND: draft + search results -> cited answer
# ============================================================================
first_query = draft["research_queries"][0]
revised = response_improver.generate(
    {
        "messages": [
            HumanMessage(content=example_question),
            initial["messages"],
            ToolMessage(
                tool_call_id=initial["messages"].tool_calls[0]["id"],
                content=json.dumps(tavily_tool.invoke({"query": first_query})),
            ),
        ]
    }
)

rev = revised["messages"].tool_calls[0]["args"]
print(f"📝 Revised answer ({len(rev['response'].split())} words): {rev['response'][:400]}...")
print(f"📚 Sources: {rev['sources']}")

📝 Revised answer (180 words): Supervised and unsupervised learning are two essential paradigms in machine learning, each with distinct characteristics. 

**Supervised Learning** involves training a model on a labeled dataset, where each input is associated with a corresponding output label. The primary objective is to learn a mapping from inputs to outputs, allowing the model to make predictions on unseen data. Common algorith...
📚 Sources: ['https://www.v7labs.com/blog/supervised-vs-unsupervised-learning', 'https://www.geeksforgeeks.org/machine-learning/supervised-unsupervised-learning', 'https://aws.amazon.com/compare/the-difference-between-machine-learning-supervised-and-unsupervised']


---
## 🔄 Part 4: The Reflexion Loop as a Graph

### 4.1 The Tool Node

Both schemas are "tools" the model calls. A `ToolNode` maps each schema name to a function that **runs the research queries** from the call's arguments - that is how the critique turns into evidence.

In [10]:
# ============================================================================
# TOOL NODE: Executing a GenerateResponse/ImproveResponse call = running its searches
# ============================================================================
from langchain_core.tools import StructuredTool
from langgraph.prebuilt import ToolNode

search_log: list[str] = []


def execute_search_queries(research_queries: list[str], **kwargs):
    """Execute the generated search queries."""
    search_log.extend(research_queries)
    return tavily_tool.batch([{"query": search_term} for search_term in research_queries])


search_executor = ToolNode(
    [
        StructuredTool.from_function(execute_search_queries, name=GenerateResponse.__name__),
        StructuredTool.from_function(execute_search_queries, name=ImproveResponse.__name__),
    ]
)
print("✅ search_executor ready")

✅ search_executor ready


### 4.2 Building the Graph

`create_draft → search_and_research → enhance_response`, then a router decides whether to research and revise again. The original router counted trailing tool/AI messages, which hid the actual limit; this one counts **revisions** directly and stops at `MAX_REVISIONS`.

In [11]:
# ============================================================================
# GRAPH: draft -> search -> revise -> (search -> revise)* -> END
# ============================================================================
from typing import Annotated

from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from typing_extensions import TypedDict

MAX_REVISIONS = 2


class State(TypedDict):
    messages: Annotated[list, add_messages]


def count_revisions(messages: list) -> int:
    return sum(
        1 for m in messages
        if m.type == "ai" and m.tool_calls and m.tool_calls[0]["name"] == ImproveResponse.__name__
    )


def determine_next_action(state: State):
    """Pure router: research and revise again, or stop after MAX_REVISIONS."""
    return END if count_revisions(state["messages"]) >= MAX_REVISIONS else "search_and_research"


def build_reflexion_workflow(responder, improver):
    workflow_builder = StateGraph(State)
    workflow_builder.add_node("create_draft", responder.generate)
    workflow_builder.add_node("search_and_research", search_executor)
    workflow_builder.add_node("enhance_response", improver.generate)

    workflow_builder.add_edge(START, "create_draft")
    workflow_builder.add_edge("create_draft", "search_and_research")
    workflow_builder.add_edge("search_and_research", "enhance_response")
    workflow_builder.add_conditional_edges("enhance_response", determine_next_action, ["search_and_research", END])
    return workflow_builder.compile()


reflexion_workflow = build_reflexion_workflow(initial_responder, response_improver)
print("🧩 Nodes:", [n for n in reflexion_workflow.get_graph().nodes if not n.startswith("__")])

🧩 Nodes: ['create_draft', 'search_and_research', 'enhance_response']


---
## 🧪 Part 5: Running and Checking the Agent

The agent iterates draft → critique → research → revision until it reaches `MAX_REVISIONS`. Each step prints a one-line summary; the last cell checks the final answer against the guidelines **in code** - word budget, numbered citations and a non-empty source list - rather than trusting the model's own claim to have followed them.

In [12]:
# ============================================================================
# RUN: Stream the Reflexion loop
# ============================================================================
import re

target_question = "How do neural networks actually learn?"
search_log.clear()

final_messages, previous_answer = None, None
for update in reflexion_workflow.stream({"messages": [("user", target_question)]}, stream_mode="updates"):
    for node, value in update.items():
        msgs = value["messages"] if isinstance(value["messages"], list) else [value["messages"]]
        if node == "search_and_research":
            print(f"🌐 {node}: ran {len(msgs)} search batch(es)")
            continue
        args = msgs[-1].tool_calls[0]["args"]
        citations = sorted(set(re.findall(r"\[\d+\]", args["response"])))
        changed = "" if previous_answer is None else f" | changed: {args['response'] != previous_answer}"
        print(f"📝 {node}: {len(args['response'].split())} words | citations: {citations}{changed}")
        print(f"   missing: {args['reflection']['missing'][:120]}")
        print(f"   next queries: {args['research_queries']}")
        final_messages, previous_answer = msgs, args["response"]

print(f"\n📊 {len(search_log)} searches in total")

📝 create_draft: 268 words | citations: []
   missing: The explanation could benefit from a more detailed discussion on different types of neural networks (e.g., CNNs, RNNs) a
   next queries: ['How do different types of neural networks learn?', 'What are activation functions in neural networks?', 'Understanding backpropagation in neural networks']


🌐 search_and_research: ran 1 search batch(es)


📝 enhance_response: 207 words | citations: ['[1]', '[2]', '[3]'] | changed: True
   missing: The explanation could include a brief mention of different types of neural networks (e.g., CNNs, RNNs) and their specifi
   next queries: ['Different types of neural networks and their learning mechanisms', 'Hyperparameters in neural network training', 'Real-world applications of neural networks']


🌐 search_and_research: ran 1 search batch(es)


📝 enhance_response: 224 words | citations: ['[1]', '[2]', '[3]'] | changed: True
   missing: The explanation could elaborate on different types of neural networks (such as CNNs and RNNs) and how they utilize simil
   next queries: ['Comparison of learning mechanisms in various neural networks', 'Role of hyperparameters in neural network performance', 'Applications of neural networks in different industries']

📊 6 searches in total


In [13]:
# ============================================================================
# CHECK THE FINAL ANSWER IN CODE
# ============================================================================
import re

final = final_messages[-1].tool_calls[0]["args"]
answer = final["response"]
body = answer.split("References")[0]
checks = {
    "within ~250 words (body <= 300)": len(body.split()) <= 300,
    "has numbered citations like [1]": bool(re.search(r"\[\d+\]", answer)),
    "lists at least one source URL": any(s.startswith("http") for s in final.get("sources", [])),
}
for name, ok in checks.items():
    print(f"{'✅' if ok else '❌'} {name}")

print("\n=== FINAL ANSWER ===")
print(answer)
print("\nSources:", *final.get("sources", []), sep="\n - ")

✅ within ~250 words (body <= 300)
✅ has numbered citations like [1]
✅ lists at least one source URL

=== FINAL ANSWER ===
Neural networks learn through a systematic process known as training, which involves adjusting the weights of connections based on input data. Here’s how the learning process works:

1. **Initialization**: The neural network begins with random weights assigned to its connections, establishing the initial influence of one neuron on another.

2. **Forward Propagation**: When input data enters the network, it flows through multiple layers of neurons. Each neuron applies an activation function to the weighted sum of its inputs, producing an output that feeds into the next layer. This continues until the output layer generates predictions.

3. **Loss Calculation**: After predictions are made, the network measures the loss, which quantifies the difference between the predicted output and the actual target values. Common loss functions include mean squared error and cross-

---
## ☁️ Part 6 (Optional): The Same Responder on Qwen via Databricks

Nothing in the responder is OpenAI-specific: `make_responder` takes any chat model that supports tool calling. This cell builds the initial responder on **Qwen 3.5 (`system.ai.qwen35-122b-a10b`) via Databricks Foundation Model APIs** and asks for one structured draft. It passes no `strict=True` (that is OpenAI-only), so the `AdaptiveResponder` retry loop is what catches a draft that misses a required field.

It runs only when `DATABRICKS_HOST` and `DATABRICKS_TOKEN` are set; otherwise it prints a skip line.

In [14]:
# ============================================================================
# DATABRICKS RESPONDER (OPTIONAL): Same schema and prompt, Qwen as the actor
# ============================================================================
import os

from aied_helpers import get_databricks_llm

DATABRICKS_RESPONDER_MODEL = "system.ai.qwen35-122b-a10b"

if not (os.getenv("DATABRICKS_HOST") and os.getenv("DATABRICKS_TOKEN")):
    print("⏭️  Skipped: DATABRICKS_HOST / DATABRICKS_TOKEN not set - the Databricks responder demo needs them")
else:
    try:
        qwen_llm = get_databricks_llm(model=DATABRICKS_RESPONDER_MODEL)
        qwen_responder = make_responder(qwen_llm, GenerateResponse, FIRST_DRAFT_INSTRUCTION)
        qwen_draft = qwen_responder.generate({"messages": [HumanMessage(content=example_question)]})
        args = qwen_draft["messages"].tool_calls[0]["args"]
        print(f"📝 Qwen's answer ({len(args['response'].split())} words): {args['response'][:300]}...")
        print(f"🌐 Qwen's research queries: {args['research_queries']}")
    except Exception as exc:  # optional demo: report, don't crash the notebook
        print(f"⚠️ Databricks responder failed: {type(exc).__name__}: {str(exc)[:200]}")

📝 Qwen's answer (226 words): Supervised and unsupervised learning are two fundamental paradigms in machine learning, distinguished primarily by the type of data they use and their learning objectives.

**Supervised Learning** involves training a model on labeled data, where each input example is paired with a corresponding outp...
🌐 Qwen's research queries: ['supervised vs unsupervised learning code examples python scikit-learn', 'evaluation metrics for supervised and unsupervised machine learning', 'real-world applications supervised unsupervised learning industry use cases']


---
## 📝 Summary

In this notebook, we learned:

### 1. The Reflexion pattern
- Reflexion extends reflection with a critique that names what is **missing** and **superfluous**, and with research that fills the gaps - the revision is grounded in evidence, not only in the model's own opinion.

### 2. A structured responder
- Binding the output schema as a *tool* (with `tool_choice`) makes the answer a tool call that a `ToolNode` can act on. A retry loop feeds validation errors back to the model.

### 3. Research and revision
- The revisor combines draft, critique and search results, and must cite them.

### 4. The loop as a graph
- `create_draft → search_and_research → enhance_response`, looped until `MAX_REVISIONS`; a router that counts revisions makes the bound explicit.
- Check the guidelines (length, citations, sources) in code after the run.

### 5. A second model family (optional)
- `make_responder` takes any tool-calling chat model; the Qwen-on-Databricks run needs only `DATABRICKS_HOST` and `DATABRICKS_TOKEN`.

### Next Steps
- **`03_reflection_architecture.ipynb`**: a single generate → critique → refine pass with a structured critique, measured by hidden tests